# ToneFlow — full 30-stem MDB-stem-synth bench

Confirms the **93.4 % RPA** CREPE-tiny number we measured locally on the
5-stem prefix. Free-tier Colab CPU is enough; expected runtime ≈ 3.5 h
(≈ 30 min NSDF baseline + ≈ 3 h CREPE-tiny).

**Before you run:** Runtime → Change runtime type → CPU. GPU/TPU give no
speedup here; `onnxruntime-node` runs CREPE in WASM.

**Don't close the tab.** Colab disconnects ~90 min after the *last* cell
finishes. Each bench cell runs continuously, so an idle disconnect should
not fire mid-bench, but a closed tab will.

Output: two log files, `bench-mdb-nsdf.log` and `bench-mdb-crepe.log`,
saved to `/content/`. The final cell offers to copy them to your Drive.

## 1. Install Node 20

Colab ships with Node 18; the bench uses Next 16 features that need 20+.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null
!sudo apt-get install -y nodejs >/dev/null
!node --version && npm --version

## 2. Clone repo + install deps

`npm ci` uses the locked dependency tree — same versions as local.

In [ ]:
!git clone --depth 1 https://github.com/FatihBastan/ToneFlow.git /content/ToneFlow
%cd /content/ToneFlow
!npm ci --no-audit --no-fund 2>&1 | tail -5

## 3. Download MDB-stem-synth (~3 GB)

Source: [Zenodo record 1481172](https://zenodo.org/records/1481172).
If the download stalls, retry — Zenodo can throttle. Should land 230
audio stems with matching annotations.

In [ ]:
!mkdir -p bench/datasets
!wget -q --show-progress -O /tmp/MDB-stem-synth.tar.gz \
  https://zenodo.org/records/1481172/files/MDB-stem-synth.tar.gz
!tar xzf /tmp/MDB-stem-synth.tar.gz -C bench/datasets/
!rm /tmp/MDB-stem-synth.tar.gz
!ls bench/datasets/MDB-stem-synth/audio_stems | wc -l

## 4. NSDF baseline (≈ 30 min)

`--rms-gate 0.005` matches the live adaptive-floor gate that fires in
quiet rooms. Without it, the bench measures the legacy `0.018` path
and reports ~55 % RPA instead of the lifted ~70 %.

In [ ]:
!npm run bench:mdb -- ./bench/datasets/MDB-stem-synth --rms-gate 0.005 \
  2>&1 | tee /content/bench-mdb-nsdf.log

## 5. CREPE-tiny bench (≈ 3 h)

Uses `public/models/crepe-tiny.onnx` (1.9 MB, in git). Same
resample → normalize → infer → local-weighted-mean pipeline as the
browser path.

In [ ]:
!npm run bench:mdb -- ./bench/datasets/MDB-stem-synth --rms-gate 0.005 --crepe \
  2>&1 | tee /content/bench-mdb-crepe.log

## 6. Inspect the headline numbers

The summary is the last few lines of each log.

In [ ]:
!echo '=== NSDF (baseline) ===' && tail -25 /content/bench-mdb-nsdf.log
!echo && echo '=== CREPE-tiny ===' && tail -25 /content/bench-mdb-crepe.log

## 7. (Optional) Save logs to Drive

Colab wipes `/content/` when the runtime disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/toneflow-bench
!cp /content/bench-mdb-*.log /content/drive/MyDrive/toneflow-bench/
!ls -la /content/drive/MyDrive/toneflow-bench/